# AI-Powered Crop Yield Prediction & Advisory System

## Objective
This notebook builds a machine learning system that:
- Predicts crop yield based on soil and environmental factors
- Provides actionable recommendations to improve agricultural productivity

## Real-World Problem
Farmers often face:
- Uncertain crop yield
- Poor soil management decisions
- Lack of data-driven guidance

This system aims to assist in **decision-making** using AI.

## Outcome
- Predict yield (Regression problem)
- Identify key influencing factors
- Provide recommendations to improve yield

# Install and Import Libraries

In [ ]:
!pip install kagglehub

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

## Data Loading

We load the agriculture dataset using KaggleHub and inspect its structure.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("bhadramohit/agriculture-and-farming-dataset")

df = pd.read_csv(path + "/agriculture_dataset.csv")
df.head()

## Data Understanding

We analyze:
- Data types
- Missing values
- Basic statistics

In [ ]:
df.info()
df.describe()

## Feature Identification

We classify:
- Numerical features
- Categorical features
- Target variable

In [ ]:
numerical_features = df.select_dtypes(include=np.number).columns.tolist()
categorical_features = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

## Missing Value Analysis

We check and handle missing values.

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()  # or fill if needed

## Outlier Detection

Boxplots help identify extreme values.

In [ ]:
for col in numerical_features:
    plt.figure()
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

## Distribution Analysis

Understanding how data is distributed.

In [ ]:
for col in numerical_features:
    plt.figure()
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

## Correlation Heatmap

Correlation analysis is performed only on numerical features because
categorical variables cannot be directly used in correlation calculations.

This helps identify relationships between environmental factors and crop yield.

In [ ]:
# Select only numerical columns
numeric_df = df.select_dtypes(include=['number'])

plt.figure(figsize=(10,8))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap (Numerical Features Only)")
plt.show()

## Agricultural Insights (Resource-Based Analysis)

This dataset focuses on **farm management practices** rather than environmental conditions.

We analyze how:
- Fertilizer usage
- Water consumption
- Crop type
- Soil type

affect crop yield.

The goal is to derive **actionable insights** for improving productivity and efficiency.

In [ ]:
target = "yield(tons)"

## FERTILIZER VS YIELD

In [ ]:
sns.scatterplot(x=df["fertilizer_used(tons)"], y=df[target])
plt.title("Fertilizer Usage vs Yield")
plt.xlabel("Fertilizer Used (tons)")
plt.ylabel("Yield (tons)")
plt.show()

## WATER USAGE VS YIELD

In [ ]:
sns.scatterplot(x=df["water_usage(cubic meters)"], y=df[target])
plt.title("Water Usage vs Yield")
plt.xlabel("Water Usage (cubic meters)")
plt.ylabel("Yield (tons)")
plt.show()

## FARM AREA VS YIELD

In [ ]:
sns.scatterplot(x=df["farm_area(acres)"], y=df[target])
plt.title("Farm Area vs Yield")
plt.xlabel("Farm Area (acres)")
plt.ylabel("Yield (tons)")
plt.show()

## CROP TYPE VS YIELD

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x=df["crop_type"], y=df[target])
plt.xticks(rotation=45)
plt.title("Crop Type vs Yield")
plt.show()

## SOIL TYPE VS YIELD

In [ ]:
sns.boxplot(x=df["soil_type"], y=df[target])
plt.title("Soil Type vs Yield")
plt.show()

## IRRIGATION TYPE VS YIELD

In [ ]:
sns.boxplot(x=df["irrigation_type"], y=df[target])
plt.title("Irrigation Type vs Yield")
plt.show()

## SEASON VS YIELD

In [ ]:
sns.boxplot(x=df["season"], y=df[target])
plt.title("Season vs Yield")
plt.show()

## Feature Engineering

We create meaningful features that reflect **resource efficiency and productivity**.

These features help the model understand not just inputs, but how effectively they are used.

In [ ]:
# Yield per acre (productivity)
df["yield_per_acre"] = df["yield(tons)"] / df["farm_area(acres)"]

# Total input usage
df["input_intensity"] = (
    df["fertilizer_used(tons)"] +
    df["pesticide_used(kg)"]
)

# Water efficiency
df["water_efficiency"] = df["yield(tons)"] / df["water_usage(cubic meters)"]

df.head()


In [ ]:
# Advanced feature engineering

# Fertilizer per acre
df["fertilizer_per_acre"] = df["fertilizer_used(tons)"] / df["farm_area(acres)"]

# Water per acre
df["water_per_acre"] = df["water_usage(cubic meters)"] / df["farm_area(acres)"]

# Pesticide per acre
df["pesticide_per_acre"] = df["pesticide_used(kg)"] / df["farm_area(acres)"]

## Encoding Categorical Variables

Categorical features are converted into numerical form for model training.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

categorical_cols = ["crop_type", "irrigation_type", "soil_type", "season"]

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

## Feature Selection and Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=[target, "farm_id"])  # remove ID
y = df[target]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## TRAIN TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

## Model Training

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor()
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results[name] = {
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds)
    }

## MODEL COMPARISON

In [ ]:
results_df = pd.DataFrame(results).T
results_df

## Explainable AI - Feature Importance

In [ ]:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)

importance = rf.feature_importances_
feat_importance = pd.Series(importance, index=X.columns)

feat_importance.sort_values().plot(kind="barh")
plt.title("Feature Importance")
plt.show()

## AI-Based Recommendations

We convert predictions into actionable farming recommendations.

In [ ]:
def recommend(row):
    recs = []

    if row["fertilizer_per_acre"] < 0.5:
        recs.append("Increase fertilizer efficiency per acre")

    if row["water_per_acre"] < 200:
        recs.append("Improve irrigation efficiency")

    if row["pesticide_per_acre"] < 1:
        recs.append("Strengthen pest control per acre")

    if row["yield_per_acre"] < 1:
        recs.append("Optimize overall farm management practices")

    return recs

## Conclusion

### Model Performance:
- Random Forest performed best with R² ≈ 0.38
- Decision Tree underperformed due to overfitting

### Key Insight:
- Yield is moderately influenced by resource usage
- However, missing variables like weather and soil nutrients limit prediction accuracy

### Business Insight:
- Efficient use of water and fertilizer improves yield
- Resource optimization is more impactful than raw usage

### Future Improvements:
- Include weather data (rainfall, temperature)
- Add soil nutrient information (NPK)
- Use advanced models like XGBoost

## Interactive AI Advisory System

We deploy the trained model using Gradio to create a simple interface where users can input farm details and receive predictions and recommendations.

In [ ]:
!pip install shap

In [ ]:
import shap

explainer = shap.TreeExplainer(model)

In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import r2_score

# Use your trained model
model = best_rf  # or rf if not tuned

# SHAP explainer
explainer = shap.TreeExplainer(model)

def predict_yield(farm_area, fertilizer, pesticide, water):

    # -------------------------------
    # 1. Create Input Data
    # -------------------------------
    input_dict = {
        "farm_area(acres)": farm_area,
        "fertilizer_used(tons)": fertilizer,
        "pesticide_used(kg)": pesticide,
        "water_usage(cubic meters)": water,
        "crop_type": 0,
        "irrigation_type": 0,
        "soil_type": 0,
        "season": 0,
    }

    df_input = pd.DataFrame([input_dict])

    # -------------------------------
    # 2. Feature Engineering
    # -------------------------------
    df_input["yield_per_acre"] = 1  # placeholder
    df_input["input_intensity"] = fertilizer + pesticide
    df_input["water_efficiency"] = 1  # placeholder

    df_input["fertilizer_per_acre"] = fertilizer / farm_area
    df_input["water_per_acre"] = water / farm_area
    df_input["pesticide_per_acre"] = pesticide / farm_area

    # Match training feature order
    df_input = df_input[X.columns]

    # -------------------------------
    # 3. Scale Input
    # -------------------------------
    input_scaled = scaler.transform(df_input)

    # -------------------------------
    # 4. Prediction
    # -------------------------------
    prediction = model.predict(input_scaled)[0]

    # -------------------------------
    # 5. SHAP Explainability
    # -------------------------------
    shap_values = explainer.shap_values(input_scaled)
    feature_impact = dict(zip(X.columns, shap_values[0]))

    # Human-readable feature mapping
    feature_names_map = {
        "fertilizer_per_acre": "Fertilizer efficiency",
        "water_per_acre": "Water usage per acre",
        "pesticide_per_acre": "Pesticide usage per acre",
        "farm_area(acres)": "Farm size",
        "fertilizer_used(tons)": "Fertilizer used",
        "water_usage(cubic meters)": "Water usage"
    }

    # Keep only meaningful features
    valid_features = [
        "fertilizer_per_acre",
        "water_per_acre",
        "pesticide_per_acre",
        "fertilizer_used(tons)",
        "water_usage(cubic meters)",
        "farm_area(acres)"
    ]

    filtered = [(k, v) for k, v in feature_impact.items() if k in valid_features]

    top_features = sorted(filtered, key=lambda x: abs(x[1]), reverse=True)[:3]

    explanation = "\n".join([
        f"{feature_names_map.get(feat, feat)}: {'increased' if val > 0 else 'decreased'} impact ({round(val,2)})"
        for feat, val in top_features
    ])

    # -------------------------------
    # 6. Recommendations
    # -------------------------------
    recs = []

    if fertilizer / farm_area < 0.5:
        recs.append("Increase fertilizer efficiency")

    if water / farm_area < 200:
        recs.append("Improve irrigation efficiency")

    if pesticide / farm_area < 1:
        recs.append("Enhance pest control")

    # -------------------------------
    # 7. Model Confidence
    # -------------------------------
    confidence = r2_score(y_test, model.predict(X_test))

    # -------------------------------
    # 8. Final Output
    # -------------------------------
    return f"""
🌾 Predicted Yield: {round(prediction,2)} tons

📊 Model Confidence (R²): {round(confidence,2)}

🔍 Key Factors:
{explanation}

📌 Recommendations:
{recs}
"""


# -------------------------------
# 9. Gradio UI
# -------------------------------
iface = gr.Interface(
    fn=predict_yield,
    inputs=[
        gr.Number(label="Farm Area (acres)"),
        gr.Number(label="Fertilizer Used (tons)"),
        gr.Number(label="Pesticide Used (kg)"),
        gr.Number(label="Water Usage (cubic meters)")
    ],
    outputs="text",
    title="🌾 AI-Based Crop Yield Advisory System",
    description="Enter farm details to predict yield, understand key factors, and get recommendations"
)

iface.launch()